## Creating a Daily Meal Plan Using AI

In [6]:
from dotenv import load_dotenv
load_dotenv()

import os
print(bool(os.getenv('OPENAI_API_KEY')))

True


In [7]:
system_role = '''
# Identity
You are a Chef-Economist AI, specializing in high-yield, low-cost gourmet meal preparation for the Dominican Republic market.

# Core Competencies
- Professional culinary arts (flavor mapping, technique optimization, presentation).
- Micro-budgeting and cost per serving analysis.
- Domain knowledge of Dominican supermarket inventories and current pricing structures via supermercadosrd.com.

# Constraints & Rules
- All currency calculations must be presented in Dominican Pesos (DOP).
- Every recipe suggestion must include a brief "Cost Breakdown" section estimating the price of ingredients based on current supermercadosrd.com metrics.
- Prioritize zero-waste cooking methods and multi-meal ingredient utility.
'''

In [15]:
def create_meals(ingredients, kcal=2000, exact_ingredients=False,
                output_format='text', model='gpt-3.5-turbo', 
                system_role=system_role, temperature=1, budget=5000, extra=None):
    from openai import OpenAI
    client = OpenAI()

    meal_request = "3 meals for tomorrow (Breakfast, Lunch, and Dinner)"
    # Logic to handle the conditional instructions cleanly
    if exact_ingredients:
        ingredient_logic = "Use ONLY the provided ingredients, plus standard pantry staples like salt, pepper, oil, and basic spices."
        budget_logic = "Since no extra ingredients should be bought, the extra budget does not need to be spent."
    else:
        ingredient_logic = f"Feel free to incorporate the provided ingredients as a base and add other necessary items to enhance the flavor, nutritional value, or overall appeal of the recipes, staying strictly within the extra budget of RD$ {budget}."
        budget_logic = f"Provide a clear breakdown of any extra items needed, using current prices from supermercadosrd.com to ensure everything fits within the RD$ {budget} limit."
        
    prompt = f"""
    Create a healthy {meal_request}.
    
    [Available Ingredients]
    ```{ingredients}```
    
    [Financial Constraints]
    Extra Budget for missing ingredients: RD$ {budget}
    
    ### Core Instructions
    1. Ingredient Rule: {ingredient_logic}
    2. Specify the exact amount of each ingredient used in the recipes.
    3. Calorie Target: Ensure that the total daily calorie intake matches a slow/moderate target of approximately {kcal} kcal.
    4. Recipe Layout: For each meal, explain the recipe step-by-step using clear, simple sentences organized with bullet points or numbers.
    5. Metrics: For each meal, specify the total number of calories, macronutrients, and the number of servings.
    6. Titles: For each meal, provide a concise and descriptive title that summarizes the main ingredients and flavors. the title should also be a valid DALL-E prompt to generate and original image for the meal.
    7. Timing: For each recipe, explicitly indicate the cooking time.
    {'8. If possible the meals should be:' + extra if extra else ''}
    
    ### Financial Breakdown
    {budget_logic}
    
    Before answering, make sure that you have followed the instructions listed above (points 1 to 7 or 8)
    Your output must be structured exactly in the following format: {output_format}.
    The last line of you answer should be a string that contains ONLY the title of the ecipes and nothing more with a comma in between.

    Example of the last line of you answer:
    '\n White Bread with Eggs, Grilled Chicken and Vegetable, Baked Fish with rice'.
    
    """

    response = client.chat.completions.create(
        model=model,
        messages=[
            { 'role': 'system', 'content': system_role},
            { 'role': 'user', 'content': prompt}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content
    

## Running the Program

In [22]:
foods='cabagge, soy sauce,rice vinager, oats, greek yogurt, milk, dark powder chocolate, cream cheese, green banana(dominican)'
output = create_meals(ingredients=foods, model='gpt-5.5-2026-04-23', output_format='HTML and CSS', extra='spicy', exact_ingredients=False)
print(output)


<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <style>
    body {
      font-family: Arial, sans-serif;
      background: #f7f5ef;
      color: #222;
      line-height: 1.55;
      margin: 0;
      padding: 24px;
    }

    .container {
      max-width: 1100px;
      margin: auto;
      background: #fff;
      padding: 28px;
      border-radius: 16px;
      box-shadow: 0 4px 18px rgba(0,0,0,0.08);
    }

    h1 {
      color: #8b1e1e;
      margin-bottom: 8px;
    }

    h2 {
      color: #b83227;
      border-bottom: 2px solid #f0c7bd;
      padding-bottom: 6px;
      margin-top: 32px;
    }

    h3 {
      color: #333;
      margin-top: 20px;
    }

    .summary, .budget, .meal {
      background: #fff9f2;
      border: 1px solid #f1d4bb;
      border-radius: 12px;
      padding: 18px;
      margin-top: 18px;
    }

    .tag {
      display: inline-block;
      background: #b83227;
      color: #fff;
      padding: 4px 10px;
      border-radius: 999px;
      fon

In [21]:
from IPython.display import display, HTML
display(HTML(output))

Ingredient,Estimated Cost Used,Budget Impact
"Oats, 65 g",RD$ 17,"Available ingredient, RD$ 0 extra"
"Greek yogurt, 170 g",RD$ 41,"Available ingredient, RD$ 0 extra"
"Milk, 200 ml",RD$ 17,"Available ingredient, RD$ 0 extra"
"Dark powder chocolate, 10 g",RD$ 10,"Available ingredient, RD$ 0 extra"
"Honey, 12 g",RD$ 10,"Available ingredient, RD$ 0 extra"
"Banana, 100 g",RD$ 25,Extra item
"Chia seeds, 10 g",RD$ 16,Extra item
"Cinnamon and cayenne, 1.5 g total",RD$ 5,Extra item
Total estimated ingredient value,RD$ 141,Extra budget used in portion: ~RD$ 46
Ingredient,Estimated Cost Used,Budget Impact
